# P4 · Temporal Fisher features

Whole-recording Fisher vectors pool acoustic windows across time. This experiment adds the difference between the first and second halves of each recording to capture changes in its sound.

In [1]:
import os, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import soundfile as sf, librosa
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
warnings.filterwarnings("ignore")

INPUT_ROOT = Path(os.environ.get("P4_INPUT", "/kaggle/input"))
WORK       = Path(os.environ.get("P4_WORK", "/kaggle/working")); WORK.mkdir(parents=True, exist_ok=True)
N_JOBS     = int(os.environ.get("P4_JOBS", -1))
SEEDS      = [42, 2026]

train_csv = list(INPUT_ROOT.rglob("train.csv")); assert len(train_csv) == 1, train_csv
DATA = train_csv[0].parent
train_df  = pd.read_csv(DATA/"train.csv", dtype={"id": str})
test_df   = pd.read_csv(DATA/"test.csv",  dtype={"id": str})
sample_df = pd.read_csv(DATA/"sample_submission.csv", dtype={"id": str})
TRAIN_AUDIO = list(INPUT_ROOT.rglob("audio/train")); TEST_AUDIO = list(INPUT_ROOT.rglob("audio/test"))
assert len(TRAIN_AUDIO) == 1 and len(TEST_AUDIO) == 1
TRAIN_AUDIO, TEST_AUDIO = TRAIN_AUDIO[0], TEST_AUDIO[0]
EXPECTED_TEST = int(os.environ.get("P4_EXPECTED_TEST", 1592))
assert test_df["id"].is_unique and sample_df["id"].is_unique and train_df["id"].is_unique
assert set(test_df["id"]) == set(sample_df["id"]), "test.csv and sample_submission.csv ids differ"
assert len(test_df) == EXPECTED_TEST, len(test_df)
print("data dir:", DATA); print(train_df.shape, test_df.shape, sample_df.shape)
print("train wav:", len(list(TRAIN_AUDIO.glob('*.wav'))), "| test wav:", len(list(TEST_AUDIO.glob('*.wav'))))

data dir: /kaggle/input/competitions/project4
(5604, 3) (1592, 1) (1592, 2)
train wav: 5604 | test wav: 1592


## Training labels

Verified recordings provide the validation labels. The table shows how many verified and unchecked recordings belong to each category.

In [2]:
classes = np.array(sorted(train_df["label"].unique())); K = len(classes)
cls_idx = {c: i for i, c in enumerate(classes)}
verified_mask = (train_df["checked"].values == 1)
y_all_idx = train_df["label"].map(cls_idx).values
y_ver_idx = y_all_idx[verified_mask];  y_unc_idx = y_all_idx[~verified_mask]
yv, yu = y_ver_idx, y_unc_idx

tab = pd.DataFrame({
    "verified":  np.bincount(yv, minlength=K),
    "unchecked": np.bincount(yu, minlength=K)}, index=classes)
tab["total"] = tab.verified + tab.unchecked
ver_cnt = tab.verified.values.astype(float)
print("classes:", K, "| verified:", verified_mask.sum(), "| unchecked:", (~verified_mask).sum())
print("verified-per-class value counts:", tab.verified.value_counts().to_dict())
big = tab.index[tab.verified == tab.verified.max()].tolist()
print("classes with the larger verified quota:", big)
print("share of verified samples in those classes: %.1f%%   share of ALL train samples in them: %.1f%%" % (
      100*tab.loc[big,'verified'].sum()/tab.verified.sum(), 100*tab.loc[big,'total'].sum()/tab.total.sum()))
display(tab.T)

classes: 41 | verified: 620 | unchecked: 4984
verified-per-class value counts: {10: 34, 40: 7}
classes with the larger verified quota: ['C03', 'C05', 'C10', 'C14', 'C36', 'C38', 'C39']
share of verified samples in those classes: 45.2%   share of ALL train samples in them: 8.1%


,C01,C02,C03,C04,C05,C06,C07,C08,C09,C10,...,C32,C33,C34,C35,C36,C37,C38,C39,C40,C41
verified,10,10,40,10,40,10,10,10,10,40,...,10,10,10,10,40,10,40,40,10,10
unchecked,162,48,2,214,40,79,206,52,171,21,...,56,152,211,63,47,183,29,8,45,130
total,172,58,42,224,80,89,216,62,181,61,...,66,162,221,73,87,193,69,48,55,140


## Recording-level features

The reference model summarizes the spectrum and its variation over each clip. Recording IDs are stored alongside the cached arrays to preserve feature alignment.

In [3]:
print('Starting recording features', flush=True)
SR, N_FFT, HOP = 22050, 2048, 512

def count_onsets(env, sr, hop):
    # Librosa's default onset peak-picking windows, expressed in frames.
    if not np.isfinite(env).all() or not np.any(env):
        return 0
    normalized = env - env.min()
    normalized = normalized / (normalized.max() + np.finfo(env.dtype).tiny)
    pre_max = int(0.03 * sr // hop)
    pre_avg = int(0.10 * sr // hop)
    post_avg = pre_avg + 1
    wait = pre_max
    count = 0
    previous = -wait - 1
    for i, value in enumerate(normalized):
        if i <= previous + wait:
            continue
        peak_window = normalized[max(0, i-pre_max):i+1]
        avg_window = normalized[max(0, i-pre_avg):i+post_avg]
        if value == peak_window.max() and value >= avg_window.mean() + 0.07:
            count += 1
            previous = i
    return count


def extract_one(path):
    try:
        y, sr = sf.read(str(path), dtype="float32")
        if y.ndim > 1: y = y.mean(axis=1)
        dur = len(y) / sr
        if len(y) < N_FFT: y = np.pad(y, (0, N_FFT - len(y)))
        S  = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)); P2 = S ** 2
        rms  = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)
        zcr  = librosa.feature.zero_crossing_rate(y=y, frame_length=N_FFT, hop_length=HOP)
        mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
        mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
        chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
        cent = librosa.feature.spectral_centroid(S=S, sr=sr)
        bw   = librosa.feature.spectral_bandwidth(S=S, sr=sr)
        roll = librosa.feature.spectral_rolloff(S=S, sr=sr)
        con  = librosa.feature.spectral_contrast(S=S, sr=sr)
        ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
        base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
        contrast = ms(con)


        r1 = rms[0]; T = S.shape[1]
        act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
        if act.sum() < 3: act = np.ones(T, bool)
        lm  = librosa.power_to_db(librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40))
        lma, mfa = lm[:, act], mfcc[:, act]
        dm  = librosa.feature.delta(mfcc, width=9 if T >= 9 else 3, mode="nearest")
        flat = librosa.feature.spectral_flatness(S=S)[0]
        env  = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
        n_on = count_onsets(env, sr, HOP)
        lr = np.log(r1 + 1e-8)
        rich = np.concatenate([
            lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
            mfa.mean(1), mfa.std(1), dm.std(1),
            [flat.mean(), flat.std(), env.mean(), env.std()],
            np.percentile(lr, [10, 50, 90]),
            [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)]])
        return np.concatenate([base, contrast, rich]).astype(np.float32)
    except Exception as e:
        return None

def extract_all(ids, folder):
    rows = Parallel(n_jobs=N_JOBS, batch_size=8, verbose=5)(delayed(extract_one)(folder / f"{i}.wav") for i in ids)
    dim = next(r.shape[0] for r in rows if r is not None)
    bad = [i for i, r in zip(ids, rows) if r is None]
    X = np.vstack([r if r is not None else np.full(dim, np.nan, np.float32) for r in rows])
    print(f"{folder.name}: {X.shape}, failed files: {len(bad)}", bad[:5]); return X

import hashlib, inspect
FEATURE_VERSION = "v4-numpy-onset"
try:    _src = inspect.getsource(extract_one)
except Exception: _src = ""
FEAT_HASH = hashlib.md5((FEATURE_VERSION + _src + str((SR, N_FFT, HOP))).encode()).hexdigest()[:10]
FEAT_FILE = WORK / f"p4_features_{FEAT_HASH}.npz"
print("feature cache file:", FEAT_FILE.name)
if FEAT_FILE.exists():
    z = np.load(FEAT_FILE, allow_pickle=True)
    if list(z["train_ids"]) == train_df["id"].tolist() and list(z["test_ids"]) == test_df["id"].tolist():
        Xtr_all, Xte_all = z["Xtr"], z["Xte"]; print("loaded cached features", Xtr_all.shape, Xte_all.shape)
if "Xtr_all" not in globals():
    t0 = time.time()
    Xtr_all = extract_all(train_df["id"].tolist(), TRAIN_AUDIO)
    Xte_all = extract_all(test_df["id"].tolist(),  TEST_AUDIO)
    np.savez_compressed(FEAT_FILE, Xtr=Xtr_all, Xte=Xte_all,
                        train_ids=np.array(train_df["id"]), test_ids=np.array(test_df["id"]))
    print("extraction time: %.0fs" % (time.time() - t0))


Xtr_all = np.where(np.isfinite(Xtr_all), Xtr_all, np.nan); Xte_all = np.where(np.isfinite(Xte_all), Xte_all, np.nan)
med = np.nanmedian(Xtr_all, axis=0)
Xtr_all = np.where(np.isnan(Xtr_all), med, Xtr_all); Xte_all = np.where(np.isnan(Xte_all), med, Xte_all)
D = Xtr_all.shape[1]; print("total feature dim:", D)
assert D > 89

FEATS = {"F89":  (Xtr_all[:, :89], Xte_all[:, :89]),
         "Frich": (Xtr_all,        Xte_all)}
XV = {k: v[0][verified_mask] for k, v in FEATS.items()}
XU = {k: v[0][~verified_mask] for k, v in FEATS.items()}
XT = {k: v[1] for k, v in FEATS.items()}
dur_tr, dur_te = Xtr_all[:, 74], Xte_all[:, 74]
print("duration (s)  verified: %.2f | unchecked: %.2f | test: %.2f" % (dur_tr[verified_mask].mean(), dur_tr[~verified_mask].mean(), dur_te.mean()))

Starting recording features
feature cache file: p4_features_8eccaff5d1.npz


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:   35.1s
[Parallel(n_jobs=-1)]: Done 520 tasks      | elapsed:   49.4s
[Parallel(n_jobs=-1)]: Done 1240 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 2248 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 3544 tasks      | elapsed:  2.4min
/usr/local/lib/python3.13/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
[Parallel(n_jobs=-1)]: Done 5128 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done 5604 out of 5604 | elapsed:  3.5min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.


train: (5604, 319), failed files: 0 []


[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    2.2s
/usr/local/lib/python3.13/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
[Parallel(n_jobs=-1)]: Done 520 tasks      | elapsed:   12.1s
[Parallel(n_jobs=-1)]: Done 1240 tasks      | elapsed:   27.9s
[Parallel(n_jobs=-1)]: Done 1592 out of 1592 | elapsed:   34.7s finished


test: (1592, 319), failed files: 0 []
extraction time: 243s
total feature dim: 319
duration (s)  verified: 5.61 | unchecked: 8.16 | test: 5.49


## Scoring and validation

A correct category at rank one, two or three earns 1, ½ or ⅓ respectively. MAP@3 averages these scores across recordings. The two weighted estimates explore how performance changes under different assumptions about test category frequencies.

In [4]:
def map3_s(yi, P):
    top3 = np.argsort(-P, axis=1)[:, :3]
    return ((top3 == yi[:, None]) / np.array([1., 2., 3.])).sum(1)
def wmap3(yi, P, w=None): return float(np.average(map3_s(yi, P), weights=w))


tot = tab.total.values.astype(float)
pi_target = tot / tot.sum()
pi_ver    = ver_cnt / ver_cnt.sum()
w_val  = (pi_target / pi_ver)[yv]
w_unif = (1.0 / pi_ver)[yv]

def adjust(P, pi):
    Q = P * pi[None, :]; return Q / Q.sum(1, keepdims=True)
def sharpen(P, t):
    Q = np.clip(P, 1e-9, None) ** t; return Q / Q.sum(1, keepdims=True)
def fit_temp(P, yi, w=None):
    best = (1e9, 1.0)
    for t in np.linspace(0.5, 5.0, 19):
        Q = sharpen(P, t); nll = -np.average(np.log(Q[np.arange(len(yi)), yi] + 1e-12), weights=w)
        if nll < best[0]: best = (nll, t)
    return best[1]
def em_prior(P, iters=200, floor=0.25):

    pi = np.full(K, 1.0 / K)
    for _ in range(iters):
        Q = P * (pi * K)[None, :]; Q /= Q.sum(1, keepdims=True); pi = Q.mean(0)
    pi = np.clip(pi, floor / K, None); return pi / pi.sum()
def em_noise(P, yi, iters=40):

    n = len(yi); Y = np.eye(K)[yi]
    T0 = np.full((K, K), 0.5 / (K - 1)); np.fill_diagonal(T0, 0.5); T = T0.copy(); pi = np.full(K, 1.0 / K)
    for _ in range(iters):
        Q = P * (pi * K)[None, :] * T[:, yi].T
        Q /= Q.sum(1, keepdims=True)
        pi = Q.mean(0)
        Tn = Q.T @ Y + 5.0 * T0
        T = Tn / Tn.sum(1, keepdims=True)
    return T, Q, pi


def make_models(kind, n_jobs=1):

    rf = lambda n, s: RandomForestClassifier(n_estimators=n, class_weight="balanced", min_samples_leaf=2, random_state=s, n_jobs=n_jobs)
    et = lambda n, s: ExtraTreesClassifier(n_estimators=n, class_weight="balanced", min_samples_leaf=2, random_state=s, n_jobs=n_jobs)
    svm = lambda C: make_pipeline(StandardScaler(), SVC(C=C, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=42))
    lr  = lambda C: make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=3000))
    if kind == "orig":
        return [("svm", svm(3), "svc__sample_weight", .40), ("lr", lr(1.0), "logisticregression__sample_weight", .10),
                ("rf", rf(400, 42), "sample_weight", .30), ("et", et(400, 42), "sample_weight", .20)]
    if kind == "rich":
        return [("svm3", svm(3), "svc__sample_weight", .25), ("svm10", svm(10), "svc__sample_weight", .15),
                ("lr", lr(0.1), "logisticregression__sample_weight", .10),
                ("rf", rf(500, 42), "sample_weight", .25), ("et", et(500, 42), "sample_weight", .25)]
    raise ValueError(kind)

def fit_predict(Xtr, ytr, wtr, X_list, kind, n_jobs=1):
    outs = [np.zeros((len(X), K)) for X in X_list]
    for name, m, wk, a in make_models(kind, n_jobs):
        if wtr is None: m.fit(Xtr, ytr)
        else:           m.fit(Xtr, ytr, **{wk: wtr})
        for o, X in zip(outs, X_list): o[:, m.classes_] += a * m.predict_proba(X)
    return outs


def stage1(feat, kind, seed):
    Xv, Xu = XV[feat], XU[feat]
    splits = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(Xv, yv))
    def one(tr, va): return fit_predict(Xv[tr], yv[tr], None, [Xu, Xv[va]], kind, 1)
    res = Parallel(n_jobs=N_JOBS)(delayed(one)(tr, va) for tr, va in splits)
    oof = np.zeros((len(yv), K))
    for (tr, va), (Pu, Pva) in zip(splits, res): oof[va] = Pva
    return dict(splits=splits, Pu=[r[0] for r in res], oof=oof)
ST1 = {}
def get_st(feat, kind, seed):
    if (feat, kind, seed) not in ST1: ST1[(feat, kind, seed)] = stage1(feat, kind, seed)
    return ST1[(feat, kind, seed)]
def final_temp(feat, kind):

    return float(np.mean([fit_temp(get_st(feat, kind, s)["oof"], yv, w_unif) for s in SEEDS]))


def select_unc(Pu, cfg, temp):
    n = len(yu); s = Pu[np.arange(n), yu]
    if cfg["method"] == "agree":
        rank = (Pu > s[:, None]).sum(1)
        keep = (rank < cfg["max_rank"]) & (s >= cfg.get("tau", 0.0)); w = np.full(n, cfg["w"])
    else:
        T, Q, pi = em_noise(sharpen(Pu, temp), yu)
        q = Q[np.arange(n), yu]; keep = q >= cfg["q_min"]; w = cfg["w"] * q
    return keep, w

def stage2(feat, kind, st, cfg):
    Xv, Xu, oof1 = XV[feat], XU[feat], st["oof"]
    def one(tr, va, Pu):

        temp = fit_temp(oof1[tr], yv[tr], w_unif[tr]) if cfg["method"] == "em" else None
        keep, w = select_unc(Pu, cfg, temp)
        Xa = np.vstack([Xv[tr], Xu[keep]]); ya = np.concatenate([yv[tr], yu[keep]])
        wa = np.concatenate([np.ones(len(tr)), w[keep]])
        return fit_predict(Xa, ya, wa, [Xv[va]], kind, 1)[0], int(keep.sum())
    res = Parallel(n_jobs=N_JOBS)(delayed(one)(tr, va, Pu) for (tr, va), Pu in zip(st["splits"], st["Pu"]))
    oof = np.zeros((len(yv), K))
    for (tr, va), (P, nk) in zip(st["splits"], res): oof[va] = P
    return oof, float(np.mean([r[1] for r in res]))


def crossfit_cal(oof, splits):

    Pc = np.zeros_like(oof)
    for tr, va in splits:
        Pc[va] = sharpen(oof[va], fit_temp(oof[tr], yv[tr], w_unif[tr]))
    return Pc

def sim_eval(P, w, powers=(0.0, 0.5, 1.0), n_draw=30, size=1500, seed=0):


    rng = np.random.default_rng(seed); p = w / w.sum(); out = {pw: [] for pw in powers}
    for _ in range(n_draw):
        idx = rng.choice(len(yv), size=size, p=p); Pd = P[idx]; pi_hat = em_prior(Pd, iters=100)
        for pw in powers:
            out[pw].append(map3_s(yv[idx], Pd if pw == 0 else adjust(Pd, pi_hat ** pw)).mean())
    return {pw: float(np.mean(v)) for pw, v in out.items()}

def cv_report(oof, splits):
    Pc = crossfit_cal(oof, splits)
    rep = {"plain": wmap3(yv, oof)}
    rep["w_raw"] = sim_eval(oof, w_val, powers=(0.0,))[0]; rep["u_raw"] = sim_eval(oof, w_unif, powers=(0.0,))[0]
    for tag, w in (("w", w_val), ("u", w_unif)):
        d = sim_eval(Pc, w); rep[f"{tag}_cal"], rep[f"{tag}_em0.5"], rep[f"{tag}_em1.0"] = d[0.0], d[0.5], d[1.0]
    return rep

def sel(rep, corrected):
    return (rep["plain"] + (rep["w_em0.5"] + rep["u_em0.5"] if corrected else rep["w_raw"] + rep["u_raw"])) / 3

def finalize_probs(P, temp, power):

    Pc = sharpen(P, temp); pi_hat = em_prior(Pc)
    return (Pc if power == 0 else adjust(Pc, pi_hat ** power)), pi_hat

OUT = WORK / "model_files"
def write_submission(P, name):
    top3 = np.argsort(-P, axis=1)[:, :3]
    sub = pd.DataFrame({"id": test_df["id"], "label": [" ".join(classes[r]) for r in top3]})
    sub = sample_df[["id"]].merge(sub, on="id", how="left", validate="one_to_one")
    assert sub.shape == (len(sample_df), 2) and sub["label"].notna().all() and sub["id"].is_unique
    assert all(len(set(s.split())) == 3 and all(c in cls_idx for c in s.split()) for s in sub["label"])
    OUT.mkdir(exist_ok=True); sub.to_csv(OUT / name, index=False, lineterminator="\n"); print("saved", OUT / name, sub.shape); return sub
print("Categories:", K, "| feature dims:", {k: v.shape[1] for k, v in XV.items()})

Categories: 41 | feature dims: {'F89': 89, 'Frich': 319}


In [5]:
def train_final(feat, kind, cfg, temp):
    Xv, Xu, Xt = XV[feat], XU[feat], XT[feat]
    if cfg is None: return fit_predict(Xv, yv, None, [Xt], kind, N_JOBS)[0], None
    Pu, = fit_predict(Xv, yv, None, [Xu], kind, N_JOBS)
    keep, w = select_unc(Pu, cfg, temp)
    Xa = np.vstack([Xv, Xu[keep]]); ya = np.concatenate([yv, yu[keep]]); wa = np.concatenate([np.ones(len(yv)), w[keep]])
    return fit_predict(Xa, ya, wa, [Xt], kind, N_JOBS)[0], keep

## Short-window features

Two dictionaries capture patterns over approximately 0.1 and 0.4 seconds. Each dictionary contains 96 clusters learned from unchecked training audio. For each clip, a histogram records how frequently its windows resemble each cluster.

In [6]:
print('Starting short-window features', flush=True)
from sklearn.cluster import MiniBatchKMeans
from scipy.spatial.distance import cdist

def patches(path):
    audio, rate = sf.read(path, dtype='float32')
    mel = librosa.feature.melspectrogram(y=audio, sr=rate, n_fft=1024, hop_length=max(1, round(rate * .012)), n_mels=40)
    logmel = librosa.power_to_db(mel, ref=np.max).T
    views = []
    for width in (8, 32):
        hop = width // 2
        frames = np.pad(logmel, ((0, max(0, width-len(logmel))), (0, 0)), mode='edge')
        starts = list(range(0, len(frames)-width+1, hop))
        if starts[-1] != len(frames)-width:
            starts.append(len(frames)-width)
        blocks = np.stack([frames[i:i+width] for i in starts])
        half = width // 2
        views.append(np.hstack([blocks.mean(1), blocks.std(1), blocks[:,half:].mean(1)-blocks[:,:half].mean(1)]))
    return views

def read_patches(df, folder):
    return Parallel(n_jobs=N_JOBS, batch_size=8, verbose=10)(delayed(patches)(folder / (i+'.wav')) for i in df.id)

train_patches = read_patches(train_df, TRAIN_AUDIO)
test_patches = read_patches(test_df, TEST_AUDIO)
rng = np.random.default_rng(42)
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
temporal_views = []
fisher_views = []
encoded = []
residual_views = []
for view in range(2):
    print(f"Building acoustic features, window size {view+1}/2", flush=True)
    samples = []
    for i in np.flatnonzero(~verified_mask):
        frames = train_patches[i][view]
        samples.append(frames[rng.choice(len(frames), min(20,len(frames)), replace=False)])
    pool = np.vstack(samples)
    scaler = StandardScaler().fit(pool)
    dictionary = MiniBatchKMeans(n_clusters=96, batch_size=2048, n_init=3, random_state=42)
    dictionary.fit(scaler.transform(pool))
    centers = dictionary.cluster_centers_
    def encode(record):
        frames = scaler.transform(record[view])
        dist = cdist(frames, centers, 'sqeuclidean')
        # Split each window’s vote among its three closest centers.
        nearest = np.argpartition(dist, 2, axis=1)[:,:3]
        weights = 1 / np.maximum(np.take_along_axis(dist, nearest, axis=1), 1e-8)
        weights /= weights.sum(axis=1,keepdims=True)
        hist = np.bincount(nearest.ravel(), weights=weights.ravel(), minlength=len(centers))
        return np.sqrt(hist / hist.sum())
    encoded.append(np.vstack([encode(record) for record in train_patches + test_patches]))
    # A smaller vocabulary keeps the residual representation compact.
    residual_dict = MiniBatchKMeans(n_clusters=32, batch_size=2048, n_init=3, random_state=17)
    residual_dict.fit(scaler.transform(pool))
    def encode_residual(record):
        frames = scaler.transform(record[view]).astype(np.float32)
        centers_r = residual_dict.cluster_centers_
        assignment = cdist(frames, centers_r, 'sqeuclidean').argmin(axis=1)
        sums = np.zeros_like(centers_r)
        np.add.at(sums, assignment, frames - centers_r[assignment])
        sums /= np.maximum(np.linalg.norm(sums, axis=1, keepdims=True), 1e-8)
        vec = np.sign(sums.ravel()) * np.sqrt(np.abs(sums.ravel()))
        return vec / max(np.linalg.norm(vec), 1e-8)
    residual_views.append(np.stack([encode_residual(r) for r in train_patches + test_patches]))
    projection = PCA(n_components=24, whiten=True, svd_solver='randomized', random_state=81)
    projected = projection.fit_transform(scaler.transform(pool))
    mixture = GaussianMixture(32, covariance_type='diag', reg_covar=.01, max_iter=70, random_state=81)
    print("Fitting Gaussian mixture", flush=True)
    mixture.fit(projected)
    def fisher_stats(x, q):
        mass = q.sum(axis=0)[:, None]
        moments = q.T @ x
        mu = mixture.means_
        var = mixture.covariances_
        norm = len(x) * np.sqrt(mixture.weights_[:, None])
        first = (moments-mass*mu) / (norm*np.sqrt(var))
        second = ((q.T @ (x*x) - 2*mu*moments + mass*mu**2)/var - mass)/(np.sqrt(2)*norm)
        v = np.concatenate([first.ravel(),second.ravel()])
        v = np.sign(v)*np.sqrt(np.abs(v))
        return (v / max(np.linalg.norm(v),1e-8)).astype(np.float32)
    def encode_fisher(record):
        x = projection.transform(scaler.transform(record[view]))
        q = mixture.predict_proba(x)
        whole = fisher_stats(x, q)
        half = len(x) // 2
        if half == 0:
            return whole, np.zeros_like(whole)
        delta = fisher_stats(x[-half:], q[-half:]) - fisher_stats(x[:half], q[:half])
        return whole, delta / max(np.linalg.norm(delta), 1e-8)
    whole, delta = zip(*(encode_fisher(r) for r in train_patches + test_patches))
    fisher_views.append(np.stack(whole))
    temporal_views.append(np.stack(delta))
    del whole, delta
fisher_matrix = np.column_stack(fisher_views)
assert np.isfinite(fisher_matrix).all()
del fisher_views
residual = np.column_stack(residual_views)
del residual_views
# A linear kernel avoids repeatedly copying the full recording-by-feature matrix.
from sklearn.metrics.pairwise import linear_kernel
kernel = linear_kernel(residual, residual)
del residual
verified_ids = np.flatnonzero(verified_mask)
unchecked_ids = np.flatnonzero(~verified_mask)
test_ids = np.arange(len(train_df), len(train_df) + len(test_df))
fisher_kernel = linear_kernel(fisher_matrix, fisher_matrix)
del fisher_matrix

temporal_matrix = np.column_stack(temporal_views)
del temporal_views
temporal_kernel = linear_kernel(temporal_matrix, temporal_matrix)
temporal_kernel += fisher_kernel
temporal_kernel *= 0.5
del temporal_matrix

def residual_model(tr, keep, target, gram=kernel):
    ids = np.r_[verified_ids[tr], unchecked_ids]
    labels = np.r_[yv[tr], yu]
    weight = np.r_[np.ones(len(tr)), np.where(keep, .5, .08)]
    svm = SVC(kernel='precomputed', C=12, probability=True, random_state=42)
    svm.fit(gram[np.ix_(ids, ids)], labels, sample_weight=weight)
    return svm.predict_proba(gram[np.ix_(target, ids)])
acoustic = np.hstack(encoded).astype('float32')
assert np.isfinite(acoustic).all()
del train_patches, test_patches
train_a, test_a = np.split(acoustic, [len(train_df)])
av, au = train_a[verified_mask], train_a[~verified_mask]

Starting short-window features


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 144 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 200 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done 344 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done 432 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 520 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done 728 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-1)]: Done 848 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done 968 tasks      | elapsed:    5.6s
[Parallel(n_jobs=-1)]: Done 1104 tasks      | elapsed:  

Building acoustic features, window size 1/2


[Parallel(n_jobs=-1)]: Done 1544 tasks      | elapsed:    7.2s
[Parallel(n_jobs=-1)]: Done 1592 out of 1592 | elapsed:    7.3s finished


Fitting Gaussian mixture
Building acoustic features, window size 2/2
Fitting Gaussian mixture


## Paired evaluation

The reference reproduces Notebook 6. The candidate combines 70% of its predictions with 30% from the temporal Fisher SVM. Both models use the same folds and label weights. Each test recording is compared with training recordings at prediction time.


In [7]:
print('Starting paired validation', flush=True)
cfg = dict(method='agree', max_rank=1, w=0.5)
def patch_model(tr, keep, target):
    X = np.vstack([av[tr], au])
    y = np.r_[yv[tr], yu]
    # Agreement with the reference model determines the weight of unchecked labels.
    weights = np.r_[np.ones(len(tr)), np.where(keep, .5, .08)]
    model = SVC(C=12, kernel='rbf', probability=True, random_state=42)
    model.fit(X, y, sample_weight=weights)
    return model.predict_proba(target)

rows = []
for seed in [42, 2026, 913]:
    print(f'Seed {seed}: fitting reference ensemble', flush=True)
    st = get_st('Frich', 'rich', seed)
    base, _ = stage2('Frich','rich',st,cfg)
    local = np.zeros_like(base)
    fisher_oof = np.zeros_like(base)
    temporal_oof = np.zeros_like(base)
    detail = np.zeros_like(base)
    for fold, ((tr, va), teacher) in enumerate(zip(st['splits'], st['Pu']), 1):
        print(f'Seed {seed}, fold {fold}/5: fitting SVMs', flush=True)
        keep, _ = select_unc(teacher,cfg,None)
        local[va] = patch_model(tr, keep, av[va])
        fisher_oof[va] = residual_model(tr,keep,verified_ids[va],fisher_kernel)
        temporal_oof[va] = residual_model(tr, keep, verified_ids[va], temporal_kernel)
        detail[va] = residual_model(tr, keep, verified_ids[va])
    ref = .65*base + .35*local
    ref = .7*ref + .3*detail
    baseline = .7*ref + .3*fisher_oof
    for name, pred in [('notebook_6', baseline), ('temporal_blend', .7*baseline + .3*temporal_oof)]:
        rep = cv_report(pred,st['splits'])
        rows.append(dict(seed=seed,model=name,MAP3=rep['plain'],selection=sel(rep, True)))
scores = pd.DataFrame(rows).set_index(['seed','model'])
display(scores)
gain = scores.xs('temporal_blend',level='model') - scores.xs('notebook_6',level='model')
display(gain)
selected = bool((gain > 0).all().all())
print('Consistent temporal Fisher gain:', selected)

Starting paired validation
Seed 42: fitting reference ensemble
Seed 42, fold 1/5: fitting SVMs
Seed 42, fold 2/5: fitting SVMs
Seed 42, fold 3/5: fitting SVMs
Seed 42, fold 4/5: fitting SVMs
Seed 42, fold 5/5: fitting SVMs
Seed 2026: fitting reference ensemble
Seed 2026, fold 1/5: fitting SVMs
Seed 2026, fold 2/5: fitting SVMs
Seed 2026, fold 3/5: fitting SVMs
Seed 2026, fold 4/5: fitting SVMs
Seed 2026, fold 5/5: fitting SVMs
Seed 913: fitting reference ensemble
Seed 913, fold 1/5: fitting SVMs
Seed 913, fold 2/5: fitting SVMs
Seed 913, fold 3/5: fitting SVMs
Seed 913, fold 4/5: fitting SVMs
Seed 913, fold 5/5: fitting SVMs


MAP3  selection
seed model                              
42   notebook_6      0.828226   0.831314
     temporal_blend  0.815591   0.826391
2026 notebook_6      0.832258   0.832977
     temporal_blend  0.820161   0.829971
913  notebook_6      0.831452   0.832089
     temporal_blend  0.825000   0.829163

,MAP3,selection
seed,,
42,-0.012634,-0.004923
2026,-0.012097,-0.003006
913,-0.006452,-0.002926


Consistent temporal Fisher gain: False


## Final predictions

The temporal Fisher SVM enters the final blend when both validation scores improve on all three seeds. The selected ensemble uses the same unlabelled class-prior adjustment as Notebook 6.


In [8]:
print('Starting final training', flush=True)
temperature = final_temp('Frich','rich')
base, keep = train_final('Frich','rich',cfg,temperature)
all_verified = np.arange(len(yv))
local = patch_model(all_verified, keep, test_a)
raw = .65*base + .35*local
raw = .7*raw + .3*residual_model(all_verified,keep,test_ids)
raw = .7*raw + .3*residual_model(all_verified, keep, test_ids, fisher_kernel)
if selected:
    raw = .7*raw + .3*residual_model(all_verified, keep, test_ids, temporal_kernel)
final, _ = finalize_probs(raw, temperature, power=.5)
sub = write_submission(final, 'p4_selected.csv')
assert len(sub) == len(test_df) and set(sub.id) == set(test_df.id)
sub.to_csv(WORK/'submission.csv', index=False)
print('Selected model:', 'Temporal Fisher blend' if selected else 'Notebook 6 Fisher blend')


Starting final training
saved /kaggle/working/model_files/p4_selected.csv (1592, 2)
Selected model: Notebook 6 Fisher blend


In [9]:
import numpy as np

required = ("baseline", "raw", "temperature", "fisher_kernel", "kernel", "selected")
if any(name not in globals() for name in required):
    raise RuntimeError("Run this cell in the completed Temporal Fisher session.")
if selected:
    raise RuntimeError("This comparison requires the Notebook 6 fallback predictions.")

def neighbour_vote(training_rows, query_ids):
    train_ids = verified_ids[training_rows]
    labels = yv[training_rows]
    k = min(12, len(train_ids))
    votes = np.zeros((len(query_ids), K))

    for start in range(0, len(query_ids), 128):
        batch = query_ids[start:start + 128]
        ix = np.ix_(batch, train_ids)
        similarity = (kernel[ix] + fisher_kernel[ix]) / 4

        nearest = np.argpartition(-similarity, k - 1, axis=1)[:, :k]
        scores = np.take_along_axis(similarity, nearest, axis=1)
        weights = np.exp((scores - scores.max(axis=1, keepdims=True)) / 0.08)
        weights /= weights.sum(axis=1, keepdims=True)

        rows = np.arange(start, start + len(batch))[:, None]
        np.add.at(votes, (rows, labels[nearest]), weights)

    return votes

# The paired evaluation retains the out-of-fold baseline from its last seed.
if seed != 913:
    raise RuntimeError("Expected the paired evaluation to finish on seed 913.")

splits = st["splits"]
candidate = np.zeros_like(baseline)
fold_gains = []

for fold, (tr, va) in enumerate(splits, 1):
    votes = neighbour_vote(tr, verified_ids[va])
    candidate[va] = 0.85 * baseline[va] + 0.15 * votes

    reference_score = wmap3(yv[va], baseline[va])
    candidate_score = wmap3(yv[va], candidate[va])
    gain = candidate_score - reference_score
    fold_gains.append(gain)
    print(f"Fold {fold}: NB6 {reference_score:.5f} | "
          f"neighbours {candidate_score:.5f} | gain {gain:+.5f}", flush=True)

overall_gain = wmap3(yv, candidate) - wmap3(yv, baseline)
wins = np.count_nonzero(np.array(fold_gains) > 0)
print(f"\nOverall MAP@3 gain: {overall_gain:+.5f} | folds improved: {wins}/5")

if overall_gain >= 0.003 and wins >= 4:
    test_votes = neighbour_vote(np.arange(len(yv)), test_ids)
    blended = 0.85 * raw + 0.15 * test_votes
    probabilities, _ = finalize_probs(blended, temperature, power=0.5)
    write_submission(probabilities, "verified_neighbours.csv")
else:
    print("Keep Notebook 6. This run produced no candidate CSV.")

Fold 1: NB6 0.84946 | neighbours 0.85484 | gain +0.00538
Fold 2: NB6 0.80914 | neighbours 0.83468 | gain +0.02554
Fold 3: NB6 0.84543 | neighbours 0.86425 | gain +0.01882
Fold 4: NB6 0.80108 | neighbours 0.79570 | gain -0.00538
Fold 5: NB6 0.85215 | neighbours 0.84812 | gain -0.00403

Overall MAP@3 gain: +0.00806 | folds improved: 3/5
Keep Notebook 6. This run produced no candidate CSV.


In [10]:
for check_seed in (42, 2026, 913):
    cached = ST1[("Frich", "rich", check_seed)]
    reference = cached["oof"]
    blended = np.empty_like(reference)
    wins = 0

    for tr, va in cached["splits"]:
        votes = neighbour_vote(tr, verified_ids[va])
        blended[va] = 0.85 * reference[va] + 0.15 * votes
        wins += int(wmap3(yv[va], blended[va]) > wmap3(yv[va], reference[va]))

    before = wmap3(yv, reference)
    after = wmap3(yv, blended)
    print(f"Seed {check_seed}: {before:.5f} → {after:.5f} | "
          f"gain {after - before:+.5f} | folds improved {wins}/5")

Seed 42: 0.75457 → 0.78414 | gain +0.02957 | folds improved 5/5
Seed 2026: 0.74839 → 0.77016 | gain +0.02177 | folds improved 5/5
Seed 913: 0.75349 → 0.78118 | gain +0.02769 | folds improved 5/5


In [11]:
check = ST1[("Frich", "rich", 42)]
print("Seed 42: fitting the reference classifiers for five folds", flush=True)
base42, _ = stage2("Frich", "rich", check, cfg)

ref42 = np.zeros_like(base42)
blend42 = np.zeros_like(base42)

for fold, ((tr, va), teacher) in enumerate(zip(check["splits"], check["Pu"]), 1):
    print(f"Fold {fold}/5: fitting acoustic models", flush=True)
    keep42, _ = select_unc(teacher, cfg, None)
    ids = verified_ids[va]

    patch42 = patch_model(tr, keep42, av[va])
    residual42 = residual_model(tr, keep42, ids)
    fisher42 = residual_model(tr, keep42, ids, fisher_kernel)

    ref42[va] = 0.65 * base42[va] + 0.35 * patch42
    ref42[va] = 0.70 * ref42[va] + 0.30 * residual42
    ref42[va] = 0.70 * ref42[va] + 0.30 * fisher42
    blend42[va] = 0.85 * ref42[va] + 0.15 * neighbour_vote(tr, ids)

    delta = wmap3(yv[va], blend42[va]) - wmap3(yv[va], ref42[va])
    print(f"Fold {fold}/5: MAP@3 gain {delta:+.5f}", flush=True)

before = wmap3(yv, ref42)
after = wmap3(yv, blend42)
print(f"Full ensemble, seed 42: {before:.5f} → {after:.5f} | gain {after-before:+.5f}")

if abs(before - 0.828226) > 0.00001:
    print("Reference score differs from the earlier run. Check before submitting.")
elif after - before >= 0.003:
    test_votes = neighbour_vote(np.arange(len(yv)), test_ids)
    probs, _ = finalize_probs(0.85 * raw + 0.15 * test_votes, temperature, power=0.5)
    write_submission(probs, "verified_neighbours_85_15.csv")
else:
    print("Seed 42 did not confirm a gain of at least 0.003. Save the submission.")

Seed 42: fitting the reference classifiers for five folds
Fold 1/5: fitting acoustic models
Fold 1/5: MAP@3 gain +0.05242
Fold 2/5: fitting acoustic models
Fold 2/5: MAP@3 gain +0.00941
Fold 3/5: fitting acoustic models
Fold 3/5: MAP@3 gain +0.00672
Fold 4/5: fitting acoustic models
Fold 4/5: MAP@3 gain +0.00134
Fold 5/5: fitting acoustic models
Fold 5/5: MAP@3 gain +0.03091
Full ensemble, seed 42: 0.82823 → 0.84839 | gain +0.02016
saved /kaggle/working/model_files/verified_neighbours_85_15.csv (1592, 2)


In [12]:
from shutil import copyfile

candidate_path = WORK / "model_files" / "verified_neighbours_85_15.csv"
candidate = pd.read_csv(candidate_path, dtype=str, keep_default_na=False)

assert candidate.columns.tolist() == ["id", "label"]
assert candidate["id"].is_unique
assert candidate["id"].tolist() == sample_df["id"].astype(str).tolist()

valid_codes = set(train_df["label"])
for recording_id, label in candidate.itertuples(index=False, name=None):
    codes = label.split(" ")
    assert len(codes) == len(set(codes)) == 3, (recording_id, label)
    assert all(code in valid_codes for code in codes), (recording_id, label)

copyfile(candidate_path, WORK / "submission.csv")
print(f"Verified {len(candidate)} rows. Saved the 85/15 blend to submission.csv")

Verified 1592 rows. Saved the 85/15 blend to submission.csv
